# Converting and Aligning Time Zones in Pandas Time Series

This notebook demonstrates:
- how to localize datetime indexes
- how to convert between time zones with pandas
- how pandas aligns time series from different time zones using UTC internally
- how to inspect available time zones with `pytz`

The examples use a small synthetic intraday stock-style dataset so the focus stays on the time zone mechanics.

## 1) Imports

We use `pandas` for the time series work, `numpy` for simple numeric helpers, and `pytz` to inspect available time zones.

In [1]:
import pandas as pd
import numpy as np
import pytz

pd.options.display.float_format = '{:,.2f}'.format

## 2) Create a Small Intraday Price Dataset

We create 30-minute OHLCV data that looks like one US trading session.

The important point is that the index starts out **naive**, which means it has dates and clock times but no timezone information attached.

In [2]:
# A naive DatetimeIndex has no timezone metadata.
naive_index = pd.date_range(start='2024-03-18 09:30', periods=6, freq='30min')

intraday_data = pd.DataFrame(
    {
        'Open': [101.20, 101.45, 101.80, 102.05, 102.10, 102.30],
        'High': [101.50, 101.90, 102.10, 102.20, 102.45, 102.55],
        'Low': [101.10, 101.35, 101.70, 101.95, 102.00, 102.20],
        'Close': [101.40, 101.80, 102.00, 102.10, 102.35, 102.50],
        'Volume': [120000, 98000, 93000, 91000, 87000, 112000],
    },
    index=naive_index,
)

intraday_data.index.name = 'Timestamp'
intraday_data.head()

,Open,High,Low,Close,Volume
Timestamp,,,,,
2024-03-18 09:30:00,101.20,101.50,101.10,101.40,120000
2024-03-18 10:00:00,101.45,101.90,101.35,101.80,98000
2024-03-18 10:30:00,101.80,102.10,101.70,102.00,93000
2024-03-18 11:00:00,102.05,102.20,101.95,102.10,91000
2024-03-18 11:30:00,102.10,102.45,102.00,102.35,87000


## 3) Inspect the Naive Index

A naive timestamp has no timezone attached. pandas knows the date and time, but not which local clock those values belong to.

In [3]:
print('First timestamps:')
print(intraday_data.index[:4])

print('\nIndex type:')
print(type(intraday_data.index))

print('\nTimezone info:')
print(intraday_data.index.tz)

First timestamps:
DatetimeIndex(['2024-03-18 09:30:00', '2024-03-18 10:00:00',
               '2024-03-18 10:30:00', '2024-03-18 11:00:00'],
              dtype='datetime64[ns]', name='Timestamp', freq='30min')

Index type:
<class 'pandas.core.indexes.datetimes.DatetimeIndex'>

Timezone info:
None


## 4) Localize the Naive Index to New York Time

Localization means assigning a timezone to naive timestamps **without changing the clock times**.

If `09:30` is meant to represent the New York market open, we should localize the index to `America/New_York`.

In [4]:
# tz_localize assigns timezone meaning to the existing clock times.
new_york_data = intraday_data.tz_localize('America/New_York')

print(new_york_data.index[:4])
print('\nTimezone info:')
print(new_york_data.index.tz)

DatetimeIndex(['2024-03-18 09:30:00-04:00', '2024-03-18 10:00:00-04:00',
               '2024-03-18 10:30:00-04:00', '2024-03-18 11:00:00-04:00'],
              dtype='datetime64[ns, America/New_York]', name='Timestamp', freq=None)

Timezone info:
America/New_York


## 5) Convert the New York Series to Other Time Zones

Once timestamps are timezone-aware, we can convert them to other zones.

Unlike localization, conversion **does change the displayed clock time** because pandas is translating the same absolute moment into another timezone.

In [5]:
# Convert the same absolute moments into UTC and Los Angeles time.
utc_data = new_york_data.tz_convert('UTC')
to_la = new_york_data.tz_convert('America/Los_Angeles')

print('New York timestamps:')
print(new_york_data.index[:3])

print('\nUTC timestamps:')
print(utc_data.index[:3])

print('\nLos Angeles timestamps:')
print(to_la.index[:3])

New York timestamps:
DatetimeIndex(['2024-03-18 09:30:00-04:00', '2024-03-18 10:00:00-04:00',
               '2024-03-18 10:30:00-04:00'],
              dtype='datetime64[ns, America/New_York]', name='Timestamp', freq=None)

UTC timestamps:
DatetimeIndex(['2024-03-18 13:30:00+00:00', '2024-03-18 14:00:00+00:00',
               '2024-03-18 14:30:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Timestamp', freq=None)

Los Angeles timestamps:
DatetimeIndex(['2024-03-18 06:30:00-07:00', '2024-03-18 07:00:00-07:00',
               '2024-03-18 07:30:00-07:00'],
              dtype='datetime64[ns, America/Los_Angeles]', name='Timestamp', freq=None)


On `2024-03-18`, New York is on daylight saving time, so it is 4 hours behind UTC and 3 hours ahead of Los Angeles.

That means:
- `10:00 a.m.` New York corresponds to `2:00 p.m.` UTC
- `10:00 a.m.` New York corresponds to `7:00 a.m.` Los Angeles

In [6]:
# Show one easy-to-read comparison subset.
time_example = pd.DataFrame(
    {
        'New York Time': new_york_data.index[:3].astype(str),
        'UTC Time': utc_data.index[:3].astype(str),
        'Los Angeles Time': to_la.index[:3].astype(str),
    }
)

time_example

,New York Time,UTC Time,Los Angeles Time
0,2024-03-18 09:30:00-04:00,2024-03-18 13:30:00+00:00,2024-03-18 06:30:00-07:00
1,2024-03-18 10:00:00-04:00,2024-03-18 14:00:00+00:00,2024-03-18 07:00:00-07:00
2,2024-03-18 10:30:00-04:00,2024-03-18 14:30:00+00:00,2024-03-18 07:30:00-07:00


## 6) Optional: What Happens if You Try to Convert Before Localizing?

You cannot use `tz_convert()` on a naive index because pandas does not yet know which timezone the timestamps belong to.

In [7]:
try:
    intraday_data.tz_convert('UTC')
except Exception as error:
    print(type(error).__name__)
    print(error)

TypeError
Cannot convert tz-naive timestamps, use tz_localize to localize


## 7) Align New York and Los Angeles Versions

Now we keep both timezone-aware versions and concatenate them side by side.

Even though one index is displayed in New York time and the other in Los Angeles time, pandas aligns them correctly because they represent the same absolute moments.

In [8]:
new_york_labeled = new_york_data.add_suffix('_NY')
los_angeles_labeled = to_la.add_suffix('_LA')

combined_data = pd.concat([new_york_labeled, los_angeles_labeled], axis=1)
combined_data.head()

,Open_NY,High_NY,Low_NY,Close_NY,Volume_NY,Open_LA,High_LA,Low_LA,Close_LA,Volume_LA
Timestamp,,,,,,,,,,
2024-03-18 13:30:00+00:00,101.20,101.50,101.10,101.40,120000,101.20,101.50,101.10,101.40,120000
2024-03-18 14:00:00+00:00,101.45,101.90,101.35,101.80,98000,101.45,101.90,101.35,101.80,98000
2024-03-18 14:30:00+00:00,101.80,102.10,101.70,102.00,93000,101.80,102.10,101.70,102.00,93000
2024-03-18 15:00:00+00:00,102.05,102.20,101.95,102.10,91000,102.05,102.20,101.95,102.10,91000
2024-03-18 15:30:00+00:00,102.10,102.45,102.00,102.35,87000,102.10,102.45,102.00,102.35,87000


pandas performs the alignment in terms of absolute time. Internally, timezone-aware timestamps can be compared consistently because pandas uses UTC as the common reference point.

So although the displayed local times differ, the rows still line up correctly.

## 8) Inspect the Aligned Index

The combined index remains timezone-aware. It reflects absolute time consistency rather than just matching clock labels.

In [9]:
print('Combined index:')
print(combined_data.index)

print('\nCombined index timezone:')
print(combined_data.index.tz)

combined_data.head(3)

Combined index:
DatetimeIndex(['2024-03-18 13:30:00+00:00', '2024-03-18 14:00:00+00:00',
               '2024-03-18 14:30:00+00:00', '2024-03-18 15:00:00+00:00',
               '2024-03-18 15:30:00+00:00', '2024-03-18 16:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='Timestamp', freq=None)

Combined index timezone:
UTC


,Open_NY,High_NY,Low_NY,Close_NY,Volume_NY,Open_LA,High_LA,Low_LA,Close_LA,Volume_LA
Timestamp,,,,,,,,,,
2024-03-18 13:30:00+00:00,101.20,101.50,101.10,101.40,120000,101.20,101.50,101.10,101.40,120000
2024-03-18 14:00:00+00:00,101.45,101.90,101.35,101.80,98000,101.45,101.90,101.35,101.80,98000
2024-03-18 14:30:00+00:00,101.80,102.10,101.70,102.00,93000,101.80,102.10,101.70,102.00,93000


## 9) Add Readable Timestamp Columns in Two Time Zones

To make the alignment easier to inspect, we can derive two explicit columns from the combined index: one displayed in New York time and one displayed in Los Angeles time.

In [ ]:
combined_data = combined_data.copy()
combined_data['timestamp_new_york'] = combined_data.index.tz_convert('America/New_York').astype(str)
combined_data['timestamp_los_angeles'] = combined_data.index.tz_convert('America/Los_Angeles').astype(str)

combined_data[['timestamp_new_york', 'timestamp_los_angeles']].head()

## 10) Discover Available Time Zones with `pytz`

It is often useful to inspect the list of available timezone names before choosing one for your dataset.

In [ ]:
print('Number of available time zones:')
print(len(pytz.all_timezones))

print('\nSample common time zones:')
print(pytz.common_timezones[:15])

america_or_us = [zone for zone in pytz.common_timezones if ('US' in zone or 'America' in zone)]
print('\nTime zones containing US or America:')
print(america_or_us[:20])

## 11) Key Difference: `tz_localize()` vs `tz_convert()`

This is the core lesson:

- `tz_localize()` is used when timestamps are naive and you need to assign the correct timezone.
- `tz_convert()` is used after timestamps are already timezone-aware and you want to express the same absolute moments in another timezone.

Short version:
- localize = assign timezone meaning
- convert = translate to another timezone

## Summary

- Localization makes timestamps timezone-aware.
- Conversion changes the displayed timezone while preserving the same absolute moment.
- pandas aligns timezone-aware data correctly across time zones.
- UTC is the internal reference point that makes this alignment consistent.
- In financial time series, getting the timezone right is essential before merging, comparing, or resampling intraday data.